In [2]:
import pandas as pd
import time
from thor_requests.connect import Connect
from thor_requests.wallet import Wallet
from thor_requests.contract import Contract
import json
import os   


In [3]:
# config 
RPC = "https://vethor-node-test.vechaindev.com" #testnet
#RPC = "" #mainnet
connector = Connect(RPC)
SMC_VESTING_ADDRESS = os.getenv('iPrivateSaleVBVesting')

key_dict = {}
with open("../../keystore") as json_file:
  key_dict = json.load(json_file)
_owner = "0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7"
_wallet = Wallet.fromKeyStore(ks=key_dict, password='passtest')

_contract = Contract.fromFile('../abi/PrivateSaleVBVesting.json')

In [4]:
def check_beneficiary(address):
  try:
    res = connector.call(
    caller=_owner,
    contract=_contract, 
    func_name="getBeneficiary", 
    func_params=[address],
    to=SMC_VESTING_ADDRESS,
    )
    return res
  except:
    return None
  return None
    
# aaa = check_beneficiary("0x4B0897b0513fdC7C541B6d9D7E929C4e5364D2dB")
# print(aaa)
def add_beneficiary(address, amount):
  txn = connector.transact(
    wallet=_wallet,
    contract=_contract,
    func_name="addBeneficiary",
    func_params=[address,amount],
    to=SMC_VESTING_ADDRESS,
    
  )
  id = txn["id"]
  tx_id = connector.wait_for_tx_receipt(tx_id=id, timeout=20)
  return id,tx_id

# print(add_beneficiary("0x9a773a0c1710a5afd9d25eb5b0d2dca2239663e6",1300000000))
# print(connector.get_tx("0x314f4fbf3e680f01e594e72704cee7c5f0a88ce89420143a968cbc4e832a7609"))

def is_confirmed(tx_hash):
    try:
        receipt = connector.get_tx_receipt(tx_id=tx_hash)
        if str(receipt["reverted"]) == "False": ## transaction success => receipt["reverted"] == False  
            return True
        return False
    except:
        return False

# print(is_confirmed("0x9fedba3fab7b5953978f606ab6e6ca5e679e5c4683430279a1c93a6ee1d360e6"))


In [5]:
progress = []


In [6]:
# added = []

# Input the data file in csv format
data = pd.read_csv("data.csv")
for row in data.iterrows():
  address = row[1][0]
  amount = int(row[1][1])*10**18
  if amount <=0: continue
  check = check_beneficiary(address)
  if check is not None:
    if str(check['reverted']) == "False":
      print(f"beneficiary {address} is already existed, skip")
      # added.append({"address": address, "amount": amount, "status": "skip"})
      continue
  tx_id,tx_hash = add_beneficiary(address, amount)
  if tx_hash != "":
    print(f"adding beneficiary {address} , txhash: {tx_hash}")
    progress.append({"address": address, "amount": amount, "status": "added", "hash": tx_id})
  time.sleep(5)

adding beneficiary 0xE4A482E15Bd8D5cAEf13B2f0EfdE7Bf15B737929 , txhash: {'gasUsed': 144844, 'gasPayer': '0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7', 'paid': '0x1419e4817e778000', 'reward': '0x607c48d3f8a4000', 'reverted': False, 'meta': {'blockID': '0x00c57a0c571a857d2e6873ebc525eb812f320a1a68c24f62828e019210486278', 'blockNumber': 12941836, 'blockTimestamp': 1659449710, 'txID': '0x5d3fcb8b55519372c9a7c03ddadffc86209d025ce5701d6d168291393109f5c5', 'txOrigin': '0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7'}, 'outputs': [{'contractAddress': None, 'events': [{'address': '0xc652898b0b05bbe0650ad4ff09e37afdf6ec2277', 'topics': ['0x8c5be1e5ebec7d5bd14f71427d1e84f3dd0314c0f7b2291e5b200ac8c7c3b925', '0x00000000000000000000000023fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7', '0x00000000000000000000000066d68ae5806ad5650ac8b17fd62cacb788ced47f'], 'data': '0x0000000000000000000000000000000000000000000014f6bf1d7cd1707c0000'}, {'address': '0xc652898b0b05bbe0650ad4ff09e37afdf6ec2277', 'topics': ['0xddf252ad

In [7]:
final = []
for item in progress:
  if "hash" in item:
    check = is_confirmed(item["hash"])
    item["confirm"] = check
    final.append(item)
df = pd.DataFrame(final)
df.to_csv('../PrivateVesting/save.csv')
df

,address,amount,status,hash,confirm
0,0xE4A482E15Bd8D5cAEf13B2f0EfdE7Bf15B737929,1001000000000000000000,added,0x5d3fcb8b55519372c9a7c03ddadffc86209d025ce570...,True
1,0xaDF66a56f668Bd18C598af93207330273E62ccA8,1002000000000000000000,added,0x0e50ab9134e4370085efa416cb0b606d14d1f3f9fb3f...,True
2,0x4B0897b0513fdC7C541B6d9D7E929C4e5364D2dB,1003000000000000000000,added,0x6d4be7650cd3385ce964b30751e97f1778f43d49ad56...,True
3,0x817f3fb962b5b090356e953134f34e63c5a7a0ad,1004000000000000000000,added,0x8354a1961453f8ca68b2d5e0f25f1839b6cae9ec5926...,True
4,0xfaae2dddb4afc844533f214273c0d89819d728e0,1005000000000000000000,added,0x4c994a76af295b5f7380fb29837a136b66942869c36e...,True
5,0xd4601ccac345484e84f8fed6fb02a40d13108c61,1006000000000000000000,added,0x854309cbe07f4a1d215ed48868d4ea93e6ac34f92864...,True
6,0x27b6b7245f0df46f5ce35658b2aeeb97d0f64912,1007000000000000000000,added,0xd9747123db340a77e791108e42c214fb9ea2653ae12f...,True
7,0x8c330dc0f8b4039a4b4a9f386b3095f49d1dafa9,1008000000000000000000,added,0x13f9d6b76659e7db2d9269ece5e1e41ccb51e38c3569...,True
8,0x4dabdfc1d1304e99be2a5ecdb62b96ca58bdc16a,1009000000000000000000,added,0xaafdf3e6f034b4713e9be09961f6b998021f0b9e18b9...,True
9,0xb60b5208684ec09288f5e36b0d3caeceee343c82,1010000000000000000000,added,0x3b5bd459960d0f65c234d5ed983de77d391b890c219c...,True
